# Lucraftmeds team crawler (Google Colab + Crawl4AI)

This notebook lets multiple teammates crawl in parallel without writing to the same file. Each teammate must use a different `WORKER_ID`, while everyone uses the same `NUM_WORKERS`. Results are checkpointed as JSONL files in a shared Google Drive folder.

> Before running: share one Google Drive folder with the team. Each teammate should add a shortcut to that folder in **My Drive**, using the same folder name/path.

## 1. Install dependencies
Run this cell once per fresh Colab runtime.

In [ ]:
%pip install -q -U crawl4ai datasets tqdm pandas
!crawl4ai-setup


## 2. Mount Google Drive and configure this worker

For a two-person team, use `NUM_WORKERS = 2`; one person sets `WORKER_ID = 0`, and the other sets `WORKER_ID = 1`. Never let two active notebooks use the same `WORKER_ID`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

# All teammates use the same values except WORKER_ID.
NUM_WORKERS = 2
WORKER_ID = 0                 # Change to 1 for the second teammate.
DRIVE_FOLDER = Path('/content/drive/MyDrive/lucraftmeds_crawl')

MAX_ITEMS = None              # Example: 100 for a quick test.
RETRY_FAILED = False          # True retries previous errors.

if not 0 <= WORKER_ID < NUM_WORKERS:
    raise ValueError('WORKER_ID must be in range [0, NUM_WORKERS).')

RESULTS_DIR = DRIVE_FOLDER / 'results'
MERGED_DIR = DRIVE_FOLDER / 'merged'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MERGED_DIR.mkdir(parents=True, exist_ok=True)
SUCCESS_PATH = RESULTS_DIR / f'worker_{WORKER_ID:02d}_success.jsonl'
ERROR_PATH = RESULTS_DIR / f'worker_{WORKER_ID:02d}_errors.jsonl'

print(f'Worker: {WORKER_ID}/{NUM_WORKERS - 1}')
print(f'Success checkpoint: {SUCCESS_PATH}')
print(f'Error checkpoint:   {ERROR_PATH}')


## 3. Load and shard the dataset
Each URL belongs to exactly one worker when everyone uses the same configuration.

In [ ]:
from datasets import load_dataset

ds = load_dataset('AIGuruTinix/ViBioMIR', 'corpus')
urls = ds['train'].shard(
    num_shards=NUM_WORKERS,
    index=WORKER_ID,
    contiguous=True,
)
if MAX_ITEMS is not None:
    urls = urls.select(range(min(MAX_ITEMS, len(urls))))
print(f'This worker owns {len(urls):,} URLs.')


## 4. Checkpoint helpers
JSONL stores one record per line, allowing immediate append and resume after interruption.

In [ ]:
import json
from typing import Iterable

def load_doc_ids(paths: Iterable[Path]) -> set[str]:
    doc_ids = set()
    for path in paths:
        if not path.exists():
            continue
        with path.open('r', encoding='utf-8') as file:
            for line_number, line in enumerate(file, start=1):
                if not line.strip():
                    continue
                try:
                    record = json.loads(line)
                except json.JSONDecodeError:
                    print(f'Warning: skipped malformed line {line_number} in {path.name}')
                    continue
                if record.get('doc_id') is not None:
                    doc_ids.add(str(record['doc_id']))
    return doc_ids

def append_jsonl(file, record: dict) -> None:
    file.write(json.dumps(record, ensure_ascii=False, default=str) + '\n')
    file.flush()

checkpoint_paths = [SUCCESS_PATH] if RETRY_FAILED else [SUCCESS_PATH, ERROR_PATH]
completed_ids = load_doc_ids(checkpoint_paths)
print(f'Resuming with {len(completed_ids):,} processed document IDs.')


## 5. Crawl4AI configuration
One browser instance is reused for the complete worker shard.

In [ ]:
def first_metadata_value(metadata: dict, *keys):
    for key in keys:
        value = metadata.get(key)

        if isinstance(value, str) and value.strip():
            return value.strip()

    return None


In [ ]:
from urllib.parse import urlparse
from crawl4ai import AsyncWebCrawler, CrawlerRunConfig
from crawl4ai.markdown_generation_strategy import DefaultMarkdownGenerator

crawl_config = CrawlerRunConfig(
    markdown_generator=DefaultMarkdownGenerator(
        content_source='fit_html',
        options={'ignore_links': True},
    ),
)

def markdown_text(markdown_result) -> str:
    if markdown_result is None:
        return ''
    for attribute in ('fit_markdown', 'raw_markdown'):
        value = getattr(markdown_result, attribute, None)
        if value:
            return value
    return str(markdown_result)

async def crawl(crawler: AsyncWebCrawler, url: str) -> dict:
    try:
        result = await crawler.arun(url=url, config=crawl_config)
    except Exception as exc:
        return {'url': url, 'success': False, 'error': f'{type(exc).__name__}: {exc}'}

    final_url = getattr(result, 'url', None) or url
    status_code = getattr(result, 'status_code', None)
    base = {
        'url': url,
        'domain': urlparse(final_url).netloc.lower(),
        'status_code': status_code,
    }
    if not result.success:
        return {
            **base,
            'success': False,
            'error': getattr(result, 'error_message', None) or 'Unknown crawl error',
        }
    metadata = getattr(result, "metadata", None) or {}

    return {
        **base,

        "title": first_metadata_value(
            metadata,
            "title",
            "og:title",
            "twitter:title",
        ),

        "description": first_metadata_value(
            metadata,
            "description",
            "og:description",
            "twitter:description",
        ),

        "keywords": first_metadata_value(
            metadata,
            "keywords",
        ),

        "markdown": markdown_text(
            getattr(result, "markdown", None)
        ),

        "success": True,
    }


## 6. Run this worker
If Colab disconnects, rerun the preceding cells and this one; completed IDs will be skipped.

In [ ]:
from tqdm.auto import tqdm

new_successes = new_errors = skipped = 0
with (
    SUCCESS_PATH.open('a', encoding='utf-8') as success_file,
    ERROR_PATH.open('a', encoding='utf-8') as error_file,
):
    async with AsyncWebCrawler() as crawler:
        for item in tqdm(urls, desc=f'Worker {WORKER_ID} crawling'):
            doc_id = str(item['id'])
            url = item['url']
            if doc_id in completed_ids:
                skipped += 1
                continue

            result = await crawl(crawler, url)
            result['doc_id'] = doc_id
            if result['success']:
                append_jsonl(success_file, result)
                new_successes += 1
            else:
                append_jsonl(error_file, result)
                new_errors += 1
            completed_ids.add(doc_id)

print({'new_successes': new_successes, 'new_errors': new_errors, 'skipped_existing': skipped})


## 7. Inspect this worker's results (optional)

In [ ]:
import pandas as pd

success_df = pd.read_json(SUCCESS_PATH, lines=True) if SUCCESS_PATH.exists() and SUCCESS_PATH.stat().st_size else pd.DataFrame()
error_df = pd.read_json(ERROR_PATH, lines=True) if ERROR_PATH.exists() and ERROR_PATH.stat().st_size else pd.DataFrame()
print(f'Successes: {len(success_df):,}; errors: {len(error_df):,}')
display(success_df.head(3))
display(error_df.head(3))
